# Tome 3, Chapter 6: Limit Theorems (Python companion)

Exercises first, solutions at the end. The central limit theorem is the reason a Monte Carlo estimate comes with an error bar, and the reason that error bar is sometimes wrong. We compute the hundred-dice probability exactly, test confidence intervals until they break, measure the Berry–Esseen rate, recover $\sqrt{2\pi}$ from a Poisson law, invert a characteristic function numerically, count the cycles of huge permutations, and use the Kolmogorov–Smirnov statistic to catch a bad Gaussian generator.

*Before each solution, predict what the output will look like.*

In [ ]:
import math
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(6)
Phi = lambda t: 0.5 * (1 + math.erf(t / math.sqrt(2)))   # N(0,1) distribution function
phi = lambda t: math.exp(-t * t / 2) / math.sqrt(2 * math.pi)

## Exercises

**P6.1 (★) A hundred dice, exactly.** Compute $P(S>400)$ for the sum $S$ of $100$ fair dice exactly, as a fraction with denominator $6^{100}$, by convolving the distribution of one die with itself. Compare with the Gaussian approximation of Corollary 6.11, with and without the correction $400.5$.

**P6.2 (★★) When do error bars lie?** The $95\%$ interval $\bar X_n\pm1.96\,\hat\sigma_n/\sqrt n$ of Exercise 6.2 is only asymptotically right. Estimate its true coverage by simulation for exponential samples and for Bernoulli$(0.01)$ samples (a rare event), with $n=10,30,100,1000,10^4$. Which one is worse, and why?

**P6.3 (★★) The Berry–Esseen rate.** For $S_n$ binomial$(n,\frac12)$, compute $D_n=\sup_t|P(\frac{S_n-n/2}{\sqrt n/2}\le t)-\Phi(t)|$ exactly for $n$ up to $10^4$. Plot $\sqrt n\,D_n$. Compare with the bound $0.48\,\mathbb E|X-\mu|^3/\sigma^3$ of the Berry–Esseen remark. Then repeat with $p=0.01$.

**P6.4 (★) The local theorem.** For $p=0.3$ and $n=40,400,4000,40000$, compute $\max_{|x_k|\le2}\bigl|\frac{\sqrt{npq}\,P(S_n=k)}{\phi(x_k)}-1\bigr|$ (Theorem 6.12). At what rate does it go to $0$? Also look at $x_k$ near $4$.

**P6.5 (★) Stirling's constant from a Poisson law.** Check numerically the identity $\mathbb E[(n-S_n)^+]=e^{-n}\frac{n^{n+1}}{n!}$ for $S_n$ Poisson$(n)$, used in Theorem 6.13, and watch $\frac{n!}{\sqrt n(n/e)^n}$ approach $\sqrt{2\pi}$. How fast?

**P6.6 (★★) Inverting a characteristic function.** By Proposition 6.7, the Laplace density $\frac12e^{-|x|}$ has characteristic function $\frac1{1+t^2}$. Recover the density from the inversion formula of Theorem 6.8 by numerical integration, and measure the error. Then estimate the characteristic function of a sample of $10^5$ Laplace variables empirically and compare.

**P6.7 (★★) Cycles of a random permutation.** Using Feller's coupling, simulate the number $C_n$ of cycles of a random permutation of $n=10^6$ elements $10^5$ times without ever building a permutation. Compare the mean and the standard deviation with $H_n$ and $s_n$ (Exercise 6.7), and the histogram with the Gaussian. Check a few true permutations too.

**P6.8 (★★) Catching a bad generator.** An old trick generates approximately Gaussian numbers as $U_1+\dots+U_{12}-6$. Using the Kolmogorov–Smirnov statistic $\sqrt m\sup_t|F_m(t)-\Phi(t)|$ (Theorem 6.5 and the remark after it), find how large a sample $m$ must be before this generator is reliably rejected, while Box–Muller is not. Use the Kolmogorov law $P(K\le x)=1-2\sum_{k\ge1}(-1)^{k-1}e^{-2k^2x^2}$.

## Solutions

**P6.1.** Python integers are exact, so the convolution gives the true numerator.

In [ ]:
dist = [1]                                   # dist[s] = number of ways to get total s
for _ in range(100):
    new = [0] * (len(dist) + 6)
    for s, c in enumerate(dist):
        if c:
            for f in range(1, 7):
                new[s + f] += c
    dist = new
p = Fraction(sum(dist[401:]), 6**100)
sd = math.sqrt(100 * 35 / 12)
print('exact P(S > 400)        =', float(p))
print('Gaussian, with 400.5    =', 1 - Phi((400.5 - 350) / sd))
print('Gaussian, with 400      =', 1 - Phi((400 - 350) / sd))
print('numerator has', len(str(p.numerator)), 'digits')

The exact answer is $0.00151$; the corrected Gaussian gives $0.00155$, the uncorrected one $0.00171$. Three standard deviations out, the relative error of the corrected approximation is about $3\%$: the theorem controls *absolute* errors in probability, uniformly in $t$, and says nothing precise about relative errors far in the tail. That is the job of large deviations (Chapter 4) and of the local theorem.

**P6.2.** Vectorise over many repetitions at once.

In [ ]:
def coverage(sampler, mu, n, reps=20000):
    X = sampler((reps, n))
    m = X.mean(axis=1)
    s = X.std(axis=1)
    ok = np.abs(m - mu) <= 1.96 * s / math.sqrt(n)
    return ok.mean()
expo = lambda size: rng.exponential(1.0, size)
bern = lambda size: (rng.random(size) < 0.01).astype(float)
for n in [10, 30, 100, 1000, 10000]:
    reps = 20000 if n <= 1000 else 2000
    print(f'n={n:6d}  exponential: {coverage(expo, 1.0, n, reps):.3f}   Bernoulli(0.01): {coverage(bern, 0.01, n, reps):.3f}')

For exponential samples the coverage is about $0.86$ at $n=10$ and creeps up to $0.95$ (the last line uses fewer repetitions, so it is noisier). For the rare event it is a disaster at small $n$ (coverage $0.09$ at $n=10$): with $n=10$ or $30$ the sample usually contains no success at all, so $\hat\sigma_n=0$, the interval is the single point $0$, and it misses $\mu=0.01$. The CLT needs $np$ to be large, not just $n$ (here the Berry–Esseen constant $\mathbb E|X-\mu|^3/\sigma^3\approx\frac1{\sqrt{p}}=10$ is huge). This is why rare-event simulation uses importance sampling, or exact binomial (Clopper–Pearson) intervals.

**P6.3.** Since $S_n$ is integer-valued, the supremum of $|F_n-\Phi|$ is attained at a jump: compare $\Phi(t_k)$ with $F_n$ just before and at each atom $t_k$.

In [ ]:
def binom_pmf(n, p):
    k = np.arange(n + 1)
    logp = (np.array([math.lgamma(n + 1) - math.lgamma(j + 1) - math.lgamma(n - j + 1) for j in k])
            + k * math.log(p) + (n - k) * math.log1p(-p))
    return np.exp(logp)
def D(n, p):
    pmf = binom_pmf(n, p)
    t = (np.arange(n + 1) - n * p) / math.sqrt(n * p * (1 - p))
    Ph = np.array([Phi(x) for x in t])
    F = np.cumsum(pmf)
    return max(np.max(np.abs(F - Ph)), np.max(np.abs(F - pmf - Ph)))
ns = [10, 30, 100, 300, 1000, 3000, 10000]
for p in [0.5, 0.01]:
    rho = (p * (1 - p)**3 + (1 - p) * p**3) / (p * (1 - p))**1.5      # E|X-mu|^3 / sigma^3
    print(f'p={p}: Berry-Esseen bound 0.48*rho/sqrt(n) with rho={rho:.3f}')
    for n in ns:
        d = D(n, p)
        print(f'   n={n:6d}  D_n={d:.5f}  sqrt(n) D_n={math.sqrt(n) * d:.3f}  bound={0.48 * rho / math.sqrt(n):.5f}')

For $p=\frac12$, $\sqrt n\,D_n$ settles near $0.40=\frac1{\sqrt{2\pi}}$: the jump of $F_n$ at the median has size $\approx\frac{2}{\sqrt{2\pi n}}$, and $\Phi$ passes through its middle, missing each end by half of it. So the rate $\frac1{\sqrt n}$ is exact, and the bound $\frac{0.48}{\sqrt n}$ is only $20\%$ pessimistic. For $p=0.01$, $\rho\approx9.9$ and the bound is useless until $n$ is in the thousands, while the true error, though below the bound, is large too ($\sqrt n\,D_n\approx2.7$ instead of $0.4$): skewness is what slows the CLT down.

**P6.4.**

In [ ]:
p, q = 0.3, 0.7
for n in [40, 400, 4000, 40000]:
    pmf = binom_pmf(n, p)
    s = math.sqrt(n * p * q)
    k = np.arange(n + 1)
    x = (k - n * p) / s
    with np.errstate(divide='ignore'):
        ratio = np.exp(np.log(s * pmf) + x**2 / 2 + 0.5 * math.log(2 * math.pi))   # in logs: no underflow
    inside = np.abs(x) <= 2
    near4 = np.argmin(np.abs(x - 4))
    print(f'n={n:6d}  max error on |x|<=2: {np.max(np.abs(ratio[inside] - 1)):.4f}'
          f'   times sqrt(n): {math.sqrt(n) * np.max(np.abs(ratio[inside] - 1)):.2f}'
          f'   ratio at x={x[near4]:.2f}: {ratio[near4]:.3f}')

The error decays like $\frac1{\sqrt n}$, exactly the size of the cubic Taylor term $\frac{Cn}6|a-p|^3$ in the proof. At $x\approx4$ the ratio is still far from $1$ for $n=40$ ($p\neq\frac12$ makes the binomial skewed); the local theorem only promises uniformity on bounded windows of $x$.

**P6.5.** Compute the Poisson probabilities by the recursion $P(S=k)=P(S=k-1)\cdot\frac nk$ in logarithms, to avoid overflow.

In [ ]:
for n in [5, 50, 500]:
    k = np.arange(0, n + 1)
    logpk = -n + k * math.log(n) - np.array([math.lgamma(j + 1) for j in k])
    lhs = np.sum((n - k) * np.exp(logpk))
    rhs = math.exp(-n + (n + 1) * math.log(n) - math.lgamma(n + 1))
    print(f'n={n:4d}  E[(n-S)^+] = {lhs:.6f}   e^-n n^(n+1)/n! = {rhs:.6f}')
print()
for n in [1, 10, 100, 1000, 10**4, 10**5]:
    r = math.exp(math.lgamma(n + 1) - 0.5 * math.log(n) - n * math.log(n) + n)
    print(f'n={n:6d}  n!/(sqrt(n)(n/e)^n) = {r:.8f}   minus sqrt(2 pi): {r - math.sqrt(2*math.pi):.2e}   times 12n/sqrt(2pi): {(r/math.sqrt(2*math.pi) - 1) * 12 * n:.4f}')

The identity holds to all printed digits. The ratio tends to $\sqrt{2\pi}=2.50663$ with error $\approx\frac{\sqrt{2\pi}}{12n}$: the last column tends to $1$, which is the next term $e^{1/(12n)}$ of Stirling's series (Tome 0).

**P6.6.** Inversion: $f(x)=\frac1{2\pi}\int e^{-itx}\varphi(t)\dd t=\frac1\pi\int_0^\infty\frac{\cos tx}{1+t^2}\dd t$ because $\varphi$ is real and even. The integrand decays only like $\frac1{t^2}$, so cut at $T$ and add the tail approximately.

In [ ]:
T, N = 2000.0, 2_000_001
t = np.linspace(0, T, N)
dt = t[1] - t[0]
w = np.full(N, dt); w[0] = w[-1] = dt / 2          # trapezoid weights
xs = np.array([0.0, 0.5, 1.0, 2.0, 4.0])
for x in xs:
    f = np.sum(w * np.cos(t * x) / (1 + t**2)) / math.pi
    print(f'x={x}: inverted {f:.6f}   true {0.5 * math.exp(-abs(x)):.6f}')
# empirical characteristic function
Y = rng.laplace(0, 1, 10**5)
for s in [0.5, 1, 2, 5, 20]:
    emp = np.mean(np.exp(1j * s * Y))
    print(f't={s:4}: empirical {emp.real:+.4f}{emp.imag:+.4f}i   exact {1/(1+s*s):.4f}')

The inversion is accurate to six digits at $x\neq0$; at $x=0$ the error is the neglected tail $\frac1\pi\int_T^\infty\frac{\dd t}{t^2}\approx\frac1{\pi T}$. The empirical characteristic function is accurate to $\approx\frac1{\sqrt n}=0.003$ for every $t$, even where the true value is tiny: an empirical CF never decays, which is why density estimation by naive inversion of an empirical CF fails and needs smoothing, exactly the Gaussian factor $e^{-\sigma^2t^2/2}$ of the proof of Theorem 6.8.

**P6.7.** In Feller's coupling, $C_n=\sum_{j=1}^nB_j$ with independent $B_j$ Bernoulli$(\frac1j)$. Summing $10^6$ Bernoullis $10^5$ times is $10^{11}$ draws, too many. We draw $B_j$ directly for $j\le1000$ (vectorised), and replace $\sum_{1000<j\le n}B_j$ by a Poisson variable with mean $\sum_{1000<j\le n}\frac1j$: by Le Cam's inequality (proved in Chapter 9) the two laws differ in total variation by at most $\sum_{j>1000}\frac1{j^2}<10^{-3}$.

In [ ]:
n, reps = 10**6, 10**5
H = sum(1 / j for j in range(1, n + 1))
H2 = sum(1 / j**2 for j in range(1, n + 1))
sn = math.sqrt(H - H2)
small = np.zeros(reps)
for j in range(1, 1001):
    small += rng.random(reps) < 1 / j
lam = sum(1 / j for j in range(1001, n + 1))
C = small + rng.poisson(lam, reps)
print(f'H_n = {H:.3f}   s_n = {sn:.3f}')
print(f'simulated mean {C.mean():.3f}   simulated sd {C.std():.3f}')
z = (C - H) / sn
for t in [-1, 0, 1, 2]:
    print(f'P(Z <= {t}) = {np.mean(z <= t):.4f}   Phi = {Phi(t):.4f}')
def cycles(perm):
    seen = np.zeros(len(perm), bool); c = 0
    for i in range(len(perm)):
        if not seen[i]:
            c += 1; j = i
            while not seen[j]:
                seen[j] = True; j = perm[j]
    return c
print('cycle counts of 5 true random permutations of 10^6:', [cycles(rng.permutation(n)) for _ in range(5)])

Mean and standard deviation match $H_n\approx14.39$ and $s_n\approx3.57$. The Gaussian fit is visibly imperfect: $C_n$ is integer-valued with a standard deviation under $4$, so $F_n$ has jumps of size about $0.1$, and the CLT for cycles converges at rate $\frac1{\sqrt{\ln n}}$, which is very slow. Note that $\frac{C_n-\ln n}{\sqrt{\ln n}}$ (the form in Exercise 6.7) is even further off at this $n$, because $H_n-\ln n\to\gamma=0.577$ is a sizeable shift when divided by $\sqrt{\ln n}\approx3.7$: same limit, worse finite-$n$ behaviour.

**P6.8.** The sum of $12$ uniforms has mean $6$ and variance $1$, and its tails are bounded by $6$: it is very close to Gaussian in the middle and wrong in the tails. The KS statistic sees the middle, so we need a large sample.

In [ ]:
def kolmogorov_sf(x, terms=100):
    k = np.arange(1, terms + 1)
    return 2 * np.sum((-1.0)**(k - 1) * np.exp(-2 * k**2 * x**2))
def ks(sample):
    xs = np.sort(sample); m = len(xs)
    F = 0.5 * (1 + np.vectorize(math.erf)(xs / math.sqrt(2))) if m <= 10**4 else 0.5 * (1 + erf_vec(xs / math.sqrt(2)))
    return math.sqrt(m) * max(np.max(np.arange(1, m + 1) / m - F), np.max(F - np.arange(m) / m))
def erf_vec(x):
    # Abramowitz-Stegun 7.1.26 is only 1e-7 accurate; use math.erf on a grid and interpolate instead
    grid = np.linspace(-8, 8, 160001)
    vals = np.array([math.erf(g) for g in grid])
    return np.interp(x, grid, vals)
def twelve(m):
    return rng.random((m, 12)).sum(axis=1) - 6
def bm(m):
    U1, U2 = 1 - rng.random(m), rng.random(m)
    return np.sqrt(-2 * np.log(U1)) * np.cos(2 * np.pi * U2)
for m in [10**4, 10**5, 10**6, 4 * 10**6]:
    k12, kbm = ks(twelve(m)), ks(bm(m))
    print(f'm={m:8d}  sum of 12: K={k12:.3f} p-value={kolmogorov_sf(k12):.4f}    Box-Muller: K={kbm:.3f} p-value={kolmogorov_sf(kbm):.4f}')

Up to $m=10^5$ both generators pass. The true distance $\sup_t|F-\Phi|$ for the sum of twelve uniforms is $2.3\cdot10^{-3}$ (computed from the exact Irwin–Hall distribution) (its kurtosis is $-0.1$ instead of $0$), so the statistic $\sqrt m\cdot2\cdot10^{-3}$ crosses the $1\%$ critical value $1.63$ around $m\approx5\cdot10^5$. Box–Muller is exact and keeps passing at the $1\%$ level (a p-value of a few percent now and then is expected: it is uniform under the null hypothesis). The lesson for testing generators: a test only sees defects larger than $\frac1{\sqrt m}$, so a generator that is good enough for $10^5$ samples may be visibly wrong for $10^7$. (A tail test would catch the sum of twelve much sooner: it never exceeds $6$, while a Gaussian exceeds $6$ with probability $2\cdot10^{-9}$, and $|N|$ exceeds $4$ with probability $6.3\cdot10^{-5}$, against $1.7\cdot10^{-5}$ for the sum of twelve.)